# **Reflection Agent with External Knowledge Integration**

This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive.


In [16]:
%%capture
%pip install langchain-openai==0.3.10
%pip install langchain==0.3.21
%pip install openai==1.68.2
%pip install langchain-community==0.2.1
%pip install  --upgrade langgraph
%pip install langchain_community==0.3.24

In [ ]:
%pip install -U langchain-ibm langchain langchain-core langchain-community langgraph langchain-openai


In [26]:
import os
import json
import getpass
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph  # MessageGraph - Deprecated

In [41]:
from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx


openai_api_key = os.getenv("OPENAI_KEY")
whatsonx_api_key = os.getenv("WATSONX_APIKEY")
ibm_project_id = os.getenv("PROJECT_ID")

openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key = openai_api_key,
)
watsonx_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=ibm_project_id,
    api_key=whatsonx_api_key,
)

In [36]:
from langchain_community.tools import TavilySearchResults
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
#search_results = tavily_tool.invoke(sample_query)
#print(search_results)

### LLM and Prompting

In [43]:
question="Any ideas for a healthy breakfast"
response=openai_llm.invoke(question).content
print(response)

Certainly! Here are some healthy breakfast ideas to start your day energized:

1. **Oatmeal with Fresh Fruits and Nuts**  
   - Whole rolled oats topped with berries, sliced banana, and a sprinkle of almonds or walnuts.  
   - Add a drizzle of honey or a sprinkle of cinnamon for extra flavor.

2. **Greek Yogurt Parfait**  
   - Layers of Greek yogurt, granola, and mixed fruits like berries, kiwi, or mango.  
   - Optional: a teaspoon of chia seeds or flaxseeds for added omega-3 fatty acids.

3. **Veggie Omelette**  
   - Eggs or egg whites cooked with spinach, tomatoes, bell peppers, and mushrooms.  
   - Serve with a side of whole-grain toast.

4. **Smoothie Bowl**  
   - Blended frozen berries, banana, and spinach topped with sliced fruits, nuts, and seeds.  
   - Use unsweetened almond or oat milk for blending.

5. **Whole Grain Toast with Avocado and Egg**  
   - Toasted whole grain bread topped with mashed avocado and a poached or scrambled egg.  
   - Add a sprinkle of salt, pepp

In [44]:
question="Any ideas for a healthy breakfast"
response=watsonx_llm.invoke(question).content
print(response)

Certainly, here are a few suggestions for a healthy breakfast:

1. **Oatmeal:** Made from whole grain oats, which are loaded with healthy fibers, vitamins, and minerals. You can top it with fruits, nuts, and a drizzle of honey for added taste and nutrition.

2. **Greek Yogurt with Nuts and Berries:** Greek yogurt is high in protein and calcium. You can add some berries for antioxidants and nuts for a dose of healthy fats and protein.

3. **Avocado on Whole Grain Toast:** Avocados are an excellent source of healthy fats and fiber. Pair it with whole grain toast for added fibers and complex carbohydrates that can give you sustained energy throughout the day.

4. **Egg and Veggie Scramble:** Eggs are a fantastic source of protein. Add in some vegetables like spinach, bell peppers, and tomatoes for a nutrient-rich meal.

5. **Smoothie:** Blend various fruits like bananas, berries, mangoes, etc., with some Greek yogurt and a handful of spinach or kale. You can also add some chia seeds or fl

#### Crafting the Agent's Persona and Logic

In [ ]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])